# 02 — Feature Engineering
## Classical vs Multi-Agent · Reply × LUISS 2026

**Objective:** Aggregate the clean datasets by **ROUTE** (departure_airport → arrival_airport),
build meaningful features and produce `features_classical.csv` ready for detection.

---

### Aggregation strategy

| Dataset | Raw granularity | Aggregation unit |
|---------|----------------|-----------------|
| `allarmi_clean.csv` | 1 row = 1 occurrence of 1 metric type per route/date | **ROUTE** |
| `viaggiatori_clean.csv` | 1 row = 1 traveller record per route/date | **ROUTE** |

> **Why not a row-by-row join?**  
> ALLARMI is a pivot table with ~30 OCCORRENZE types per route/date → granularity incompatible
> with VIAGGIATORI. The correct join happens at the aggregation level: first aggregate each dataset
> separately, then do an **outer join** on the 567 distinct routes.

---
**Output:**
- `data/processed/features_classical.csv` — 567 routes × 63 features
- `data/processed/feature_cols.json` — list of numerical features for ML models


In [1]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# Paths — works from both notebooks/ and the root
ROOT     = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

print(f"ROOT:     {ROOT}")
print(f"PROC_DIR: {PROC_DIR}")

ROOT:     /Users/fil/Desktop/Machine learning condiviso /classical-vs-multiagent
PROC_DIR: /Users/fil/Desktop/Machine learning condiviso /classical-vs-multiagent/data/processed


## 1. Loading clean data


In [2]:
allarmi     = pd.read_csv(PROC_DIR / "allarmi_clean.csv")
viaggiatori = pd.read_csv(PROC_DIR / "viaggiatori_clean.csv")

print(f"ALLARMI clean:     {allarmi.shape[0]:>5} rows × {allarmi.shape[1]:>2} col")
print(f"VIAGGIATORI clean: {viaggiatori.shape[0]:>5} rows × {viaggiatori.shape[1]:>2} col")

ALLARMI clean:      5080 righe × 20 col
VIAGGIATORI clean:  5095 righe × 31 col


## 2. Building the ROUTE key

The key is `AREOPORTO_PARTENZA-AREOPORTO_ARRIVO` (IATA codes uppercase, already normalised in preprocessing).


In [3]:
allarmi["ROTTA"]     = (allarmi["AREOPORTO_PARTENZA"].str.upper()
                        + "-" + allarmi["AREOPORTO_ARRIVO"].str.upper())
viaggiatori["ROTTA"] = (viaggiatori["AREOPORTO_PARTENZA"].str.upper()
                        + "-" + viaggiatori["AREOPORTO_ARRIVO"].str.upper())

rotte_a = set(allarmi["ROTTA"].unique())
rotte_v = set(viaggiatori["ROTTA"].unique())
comuni  = rotte_a & rotte_v

print(f"Rotte distinte in ALLARMI:     {len(rotte_a)}")
print(f"Rotte distinte in VIAGGIATORI: {len(rotte_v)}")
print(f"Rotte in comune:               {len(comuni)}")
print(f"Solo in ALLARMI:               {len(rotte_a - rotte_v)}")
print(f"Solo in VIAGGIATORI:           {len(rotte_v - rotte_a)}")

Rotte distinte in ALLARMI:     368
Rotte distinte in VIAGGIATORI: 467
Rotte in comune:               268
Solo in ALLARMI:               100
Solo in VIAGGIATORI:           199


## 3. Helper functions


In [4]:
def safe_mode(x):
    """Mode of a pandas series; returns 'ND' if the series is empty."""
    m = x.mode()
    return m.iloc[0] if len(m) > 0 else "ND"

def safe_div(a, b):
    """Vectorised safe division: returns 0.0 where b == 0."""
    return np.where(b > 0, a / b, 0.0)

print("Helper functions defined")

✓ Funzioni helper definite


## 4. Aggregation of ALLARMI by ROUTE

### 4.1 Pivot OCCORRENZE → numerical columns

ALLARMI has ~30 OCCORRENZE types per route (e.g. "Allarmi generati da INTERPOL", "Voli con Allarmi", etc.).
We do a **pivot_table**: each OCCORRENZA type becomes a numerical column (sum of TOT) per ROUTE.


In [5]:
# Pivot: one column per OCCORRENZA type, value = sum of TOT per ROUTE
occ_pivot = allarmi.pivot_table(
    index      = "ROTTA",
    columns    = "OCCORRENZE",
    values     = "TOT",
    aggfunc    = "sum",
    fill_value = 0
).reset_index()
occ_pivot.columns.name = None

# Rename to readable snake_case
RENAME_OCC = {
    "Viaggiatori entrati nel Sistema"             : "vg_entrati_occ",
    "Viaggiatori con Allarmi"                     : "vg_con_allarmi",
    "Viaggiatori investigati"                     : "vg_investigati_occ",
    "Voli con Allarmi"                            : "voli_con_allarmi",
    "Voli disponibili in ingresso al Sistema"     : "voli_disponibili",
    "Voli investigati (SDI/NSIS - INTERPOL - TSC)": "voli_investigati",
    "Voli solo visualizzati, ma NON investigati"  : "voli_non_investigati",
    "Allarmi generati"                            : "allarmi_generati",
    "Allarmi generati da SDI/NSIS"                : "allarmi_sdi_occ",
    "Allarmi generati da INTERPOL"                : "allarmi_interpol_occ",
    "Allarmi generati da BCS"                     : "allarmi_bcs_occ",
    "Allarmi Chiusi"                              : "allarmi_chiusi",
    "Allarmi Chiusi con Azione (CC.xx)"           : "allarmi_chiusi_azione",
    "Allarmi NON Chiusi"                          : "allarmi_non_chiusi",
    "Allarmi Rilevanti"                           : "allarmi_rilevanti",
    "Respinto/a"                                  : "vg_respinti_occ",
    "Errata segnalazione SDI"                     : "err_sdi",
    "Errata segnalazione NSIS"                    : "err_nsis",
    "Errata segnalazione BCS"                     : "err_bcs",
    "Nulla a procedere SDI"                       : "np_sdi",
    "Nulla a procedere NSIS"                      : "np_nsis",
    "Nulla a procedere INT"                       : "np_int",
    "Notifica Atti/Provv"                         : "notifica_atti",
    "Mancato aggiornamento SDI"                   : "mancato_agg_sdi",
    "Mancato aggiornamento Schengen NSIS"         : "mancato_agg_nsis",
    "Inammissibilita Schengen - Art.24"           : "inammissib_schengen",
    "ALLARMATI"                                   : "allarmati_occ",
    "Altro"                                       : "altro_occ",
    "N/C"                                         : "nc_occ",
    "???"                                         : "unknown_occ",
}
occ_pivot = occ_pivot.rename(columns=RENAME_OCC)

print(f"occ_pivot: {occ_pivot.shape[0]} rotte × {occ_pivot.shape[1]} colonne")

occ_pivot: 368 rotte × 28 colonne


### 4.2 % alarm reason features from MOTIVO_ALLARME

`pct_interpol`, `pct_sdi`, etc. are calculated from `MOTIVO_ALLARME` (743 INTERPOL records)  
rather than from OCCORRENZE="Allarmi generati da INTERPOL" (only 8 records) — much richer data.


In [6]:
# Counts per reason per route
motivo_counts = (
    allarmi.groupby("ROTTA")["MOTIVO_ALLARME"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)
motivo_counts.columns.name = None

# Normalised percentages [0,1]
motivo_totale = motivo_counts.drop(columns=["ROTTA"]).sum(axis=1)
for col in ["INTERPOL", "SDI", "NSIS", "TSC", "Manuale"]:
    if col in motivo_counts.columns:
        motivo_counts[f"pct_{col.lower()}"] = safe_div(
            motivo_counts[col], motivo_totale
        ).clip(0, 1)
        motivo_counts = motivo_counts.drop(columns=[col])

print(f"motivo_counts: {motivo_counts.shape}")
print("\npct_interpol statistics:")
print(motivo_counts["pct_interpol"].describe().to_string())

motivo_counts: (351, 6)

Statistiche pct_interpol:
count   351.0000
mean      0.1857
std       0.2247
min       0.0000
25%       0.0000
50%       0.1471
75%       0.2500
max       1.0000


### 4.3 Route metadata + log-transform of tot_allarmi


In [7]:
# Metadata: geographic zone, country, total alarm volume
meta_allarmi = allarmi.groupby("ROTTA").agg(
    ZONA                   = ("ZONA",      "first"),
    PAESE_PART             = ("PAESE_PART", "first"),
    n_osservazioni_allarmi = ("TOT",        "count"),
    tot_allarmi_sum        = ("TOT",        "sum"),
).reset_index()

# ── FIX: Log-transform for tot_allarmi_sum (max=103254 → skewed distribution) ──
# log1p(x) = log(1+x) handles x=0 without issues
meta_allarmi["tot_allarmi_log"] = np.log1p(meta_allarmi["tot_allarmi_sum"])

print("tot_allarmi_sum (raw):")
print(meta_allarmi["tot_allarmi_sum"].describe().apply(lambda x: f"{x:,.0f}").to_string())
print("\ntot_allarmi_log (after log1p — max≈11.5):")
print(meta_allarmi["tot_allarmi_log"].describe().to_string())

tot_allarmi_sum (grezzo):
count      368
mean       552
std        757
min          0
25%         22
50%        282
75%        706
max      3,683

tot_allarmi_log (dopo log1p — max≈11.5):
count   368.0000
mean      4.8865
std       2.2701
min       0.0000
25%       3.1244
50%       5.6454
75%       6.5607
max       8.2118


### 4.4 Merge ALLARMI components + derived ratios


In [8]:
# Combine metadata + occurrences pivot + reason percentages
agg_allarmi = (
    meta_allarmi
    .merge(occ_pivot,     on="ROTTA", how="left")
    .merge(motivo_counts, on="ROTTA", how="left")
)

# ── FIX: clip negatives from OCCORRENZE pivot ────────────────────────────────
# np_sdi and voli_non_investigati have TOT < 0 in the source dataset.
# Clip to 0 on all numerics before calculating derived ratios.
_num_cols = agg_allarmi.select_dtypes(include="number").columns
agg_allarmi[_num_cols] = agg_allarmi[_num_cols].clip(lower=0)

# Alarm closure rate (closed / closed+not_closed)
agg_allarmi["tasso_chiusura"] = safe_div(
    agg_allarmi["allarmi_chiusi"],
    agg_allarmi["allarmi_chiusi"] + agg_allarmi["allarmi_non_chiusi"]
).clip(0, 1)

# Relevance rate (relevant alarms / flights with alarms)
agg_allarmi["tasso_rilevanza"] = safe_div(
    agg_allarmi["allarmi_rilevanti"],
    agg_allarmi["voli_con_allarmi"]
).clip(0, 1)

# Fill NaN in pct_ columns (routes with no MOTIVO_ALLARME recorded)
pct_cols = [c for c in agg_allarmi.columns if c.startswith("pct_")]
agg_allarmi[pct_cols] = agg_allarmi[pct_cols].fillna(0)

print(f"agg_allarmi: {agg_allarmi.shape[0]} routes × {agg_allarmi.shape[1]} columns")
print(f"  Residual nulls: {agg_allarmi.isnull().sum().sum()}")
neg_check = {c: int((agg_allarmi[c]<0).sum()) for c in _num_cols if (agg_allarmi[c]<0).sum()>0}
print(f"  Negatives after clip: {neg_check if neg_check else 'none'}  ← MUST be none")


✓ agg_allarmi: 368 rotte × 40 colonne
  Null residui: 1
  Negativi dopo clip: ✓ nessuno  ← DEVE essere nessuno


In [9]:
# ── New feature: false_positive_rate ────────────────────────────────────────
# From the partner notebook (03_Feature_Engineering.ipynb):
# Measures how many alarms turned out to be false positives (no action taken).
# Denominator: SDI/NSIS + INTERPOL alarms (those verifiable)
# Numerator:    no-action SDI + NSIS + INT

# The necessary columns are already in occ_pivot after renaming
numeratore   = (agg_allarmi.get("np_sdi",   pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("np_nsis",  pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("np_int",   pd.Series(0, index=agg_allarmi.index)).clip(lower=0))

denominatore = (agg_allarmi.get("allarmi_sdi_occ",     pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("allarmi_interpol_occ", pd.Series(0, index=agg_allarmi.index)).clip(lower=0))

agg_allarmi["false_positive_rate"] = safe_div(numeratore, denominatore).clip(0, 1)

print(f"false_positive_rate: min={agg_allarmi['false_positive_rate'].min():.4f}, "
      f"max={agg_allarmi['false_positive_rate'].max():.4f}, "
      f"mean={agg_allarmi['false_positive_rate'].mean():.4f}")
print(f"  Routes with FPR > 0.5: {(agg_allarmi['false_positive_rate'] > 0.5).sum()} "
      f"(high share of alarms that turned out to be false)")

false_positive_rate: min=0.0000, max=1.0000, mean=0.1276
  Rotte con FPR > 0.5: 43 (alta quota di allarmi rivelatisi falsi)


## 5. Aggregation of VIAGGIATORI by ROUTE

### 5.1 Base aggregation


In [10]:
agg_viaggiatori = viaggiatori.groupby("ROTTA").agg(
    tot_entrati              = ("ENTRATI",           "sum"),
    tot_allarmati            = ("ALLARMATI",          "sum"),
    tot_investigati          = ("INVESTIGATI",        "sum"),
    n_osservazioni_viag      = ("ENTRATI",            "count"),
    tasso_allarme_medio      = ("tasso_allarme",      "mean"),
    tasso_inv_medio          = ("tasso_investigati",  "mean"),
    genere_predominante      = ("GENERE",             safe_mode),
    fascia_eta_predominante  = ("FASCIA_ETA",         safe_mode),
    tipo_doc_prevalente      = ("TIPO_DOCUMENTO",     safe_mode),
    nazionalita_top          = ("NAZIONALITA",        safe_mode),
    compagnia_top            = ("COMPAGNIA_AEREA",    safe_mode),
).reset_index()

# ── QUALITY FIX: clip negatives and cap rates to [0,1] after aggregation ────
# Defensive: preprocessing already handles negatives at row level,
# but the aggregated mean can exceed [0,1] due to residual outliers
agg_viaggiatori["tot_entrati"]         = agg_viaggiatori["tot_entrati"].clip(lower=0)
agg_viaggiatori["tot_allarmati"]       = agg_viaggiatori["tot_allarmati"].clip(lower=0)
agg_viaggiatori["tot_investigati"]     = agg_viaggiatori["tot_investigati"].clip(lower=0)
agg_viaggiatori["tasso_allarme_medio"] = agg_viaggiatori["tasso_allarme_medio"].clip(0, 1)
agg_viaggiatori["tasso_inv_medio"]     = agg_viaggiatori["tasso_inv_medio"].clip(0, 1)

print(f"agg_viaggiatori: {agg_viaggiatori.shape[0]} routes × {agg_viaggiatori.shape[1]} columns")
print(f"  tasso_inv_medio: min={agg_viaggiatori['tasso_inv_medio'].min():.4f}, max={agg_viaggiatori['tasso_inv_medio'].max():.4f}")
print(f"  tot_entrati:     min={agg_viaggiatori['tot_entrati'].min():.0f}, max={agg_viaggiatori['tot_entrati'].max():.0f}")

✓ agg_viaggiatori: 467 rotte × 12 colonne
  tasso_inv_medio: min=0.0000, max=1.0000
  tot_entrati:     min=0, max=30983


### 5.2 Pivot ESITO_CONTROLLO + risk score

We count how many times each outcome appears per route.  
**Respinto** (security action taken) and **Fermato** are the strongest signals.


In [11]:
esiti_pivot = (
    viaggiatori.pivot_table(
        index      = "ROTTA",
        columns    = "ESITO_CONTROLLO",
        values     = "ENTRATI",
        aggfunc    = "count",
        fill_value = 0
    )
    .reset_index()
)
esiti_pivot.columns.name = None
esiti_pivot = esiti_pivot.rename(columns={
    "SEGNALATO" : "n_segnalati",
    "IN ATTESA"  : "n_in_attesa",
    "RESPINTO"   : "n_respinti",
    "FERMATO"    : "n_fermati",
    "OK"         : "n_ok",
})

agg_viaggiatori = agg_viaggiatori.merge(esiti_pivot, on="ROTTA", how="left")
for col in ["n_segnalati", "n_in_attesa", "n_respinti", "n_fermati", "n_ok"]:
    if col in agg_viaggiatori.columns:
        agg_viaggiatori[col] = agg_viaggiatori[col].fillna(0).astype(int)

# Outcome risk rates (normalised on n_osservazioni)
agg_viaggiatori["tasso_respinti"] = safe_div(
    agg_viaggiatori["n_respinti"], agg_viaggiatori["n_osservazioni_viag"]
)
agg_viaggiatori["tasso_fermati"] = safe_div(
    agg_viaggiatori["n_fermati"], agg_viaggiatori["n_osservazioni_viag"]
)

# Outcome risk score: respinto weights 60%, fermato 40%
agg_viaggiatori["score_rischio_esiti"] = (
    agg_viaggiatori["tasso_respinti"] * 0.6 +
    agg_viaggiatori["tasso_fermati"]  * 0.4
).clip(0, 1)

print(f"agg_viaggiatori with outcomes: {agg_viaggiatori.shape[0]} routes × {agg_viaggiatori.shape[1]} columns")
print(f"  score_rischio_esiti: min={agg_viaggiatori['score_rischio_esiti'].min():.4f}, max={agg_viaggiatori['score_rischio_esiti'].max():.4f}")
print(f"  Residual nulls: {agg_viaggiatori.isnull().sum().sum()}")

✓ agg_viaggiatori con esiti: 467 rotte × 20 colonne
  score_rischio_esiti: min=0.0000, max=0.6000
  Null residui: 0


In [12]:
# ── New feature: alarm_per_invest ────────────────────────────────────────────
# From the partner notebook (03_Feature_Engineering.ipynb):
# Ratio of alerted / investigated per route.
# Measures alarm intensity on persons actually investigated.
# Capped at p99 to avoid extreme outliers (same approach as partner notebook).

agg_viaggiatori["alarm_per_invest"] = safe_div(
    agg_viaggiatori["tot_allarmati"],
    agg_viaggiatori["tot_investigati"]
)

# Cap at percentile 99 — extreme values are almost certainly encoding errors
cap_p99 = agg_viaggiatori["alarm_per_invest"].quantile(0.99)
agg_viaggiatori["alarm_per_invest"] = agg_viaggiatori["alarm_per_invest"].clip(upper=cap_p99)

print(f"alarm_per_invest: min={agg_viaggiatori['alarm_per_invest'].min():.4f}, "
      f"max={agg_viaggiatori['alarm_per_invest'].max():.4f} (cappato a p99={cap_p99:.4f}), "
      f"mean={agg_viaggiatori['alarm_per_invest'].mean():.4f}")

alarm_per_invest: min=0.0000, max=1.0000 (cappato a p99=1.0000), mean=0.2039


## 6. Outer Join: ALLARMI ∪ VIAGGIATORI

We use an **outer join** to retain all 567 distinct routes,
including those present in only one dataset.


In [13]:
features = agg_allarmi.merge(agg_viaggiatori, on="ROTTA", how="outer")

print(f"Shape after outer join: {features.shape}")
print(f"\nRoute coverage:")
both  = (features["n_osservazioni_allarmi"] > 0) & (features["n_osservazioni_viag"] > 0)
only_a = (features["n_osservazioni_allarmi"] > 0) & features["n_osservazioni_viag"].isna()
only_v = features["n_osservazioni_allarmi"].isna() & (features["n_osservazioni_viag"] > 0)
print(f"  Both datasets:       {both.sum()}")
print(f"  Only ALLARMI:        {only_a.sum()}")
print(f"  Only VIAGGIATORI:    {only_v.sum()}")

Shape dopo outer join: (567, 61)

Copertura rotte:
  Entrambi i dataset:  268
  Solo ALLARMI:        99
  Solo VIAGGIATORI:    199


### 6.1 Handling nulls after outer join

- `ZONA` / `PAESE_PART`: null for the 201 routes only in VIAGGIATORI → fill **"ND"**
- Numerical columns: null where the route is missing from a dataset → fill **0**
- Categorical columns: residual nulls → fill **"ND"**


In [14]:
# ── FIX PAESE_PART/ZONA: use viaggiatori for routes without alarms ─────────────
# 200 routes are only in VIAGGIATORI → PAESE_PART would be ND without this fix
paese_viag = (
    viaggiatori.groupby("ROTTA")
    .agg(
        PAESE_PART_viag = ("PAESE_PART", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "ND"),
        ZONA_viag       = ("ZONA",       lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "ND"),
    )
    .reset_index()
)
features = features.merge(paese_viag, on="ROTTA", how="left")

# Priority: allarmi → viaggiatori → "ND"
features["PAESE_PART"] = (
    features["PAESE_PART"]
    .replace(["ND", "//", ""], None)
    .fillna(features["PAESE_PART_viag"])
    .fillna("ND")
)
features["ZONA"] = (
    features["ZONA"]
    .replace(["ND", "//", ""], None)
    .fillna(features["ZONA_viag"])
    .fillna("ND")
)
features = features.drop(columns=["PAESE_PART_viag", "ZONA_viag"])

# Numerical columns: fill with 0
num_cols = features.select_dtypes(include="number").columns
features[num_cols] = features[num_cols].fillna(0)

# Remaining categorical columns: fill with "ND"
cat_cols = features.select_dtypes(include="object").columns.drop("ROTTA")
for col in cat_cols:
    features[col] = features[col].fillna("ND")

null_total = features.isnull().sum().sum()
nd_paese   = (features["PAESE_PART"] == "ND").sum()
neg_cols   = {c: int((features[c] < 0).sum()) for c in features.select_dtypes("number").columns if (features[c] < 0).any()}

print(f"Total nulls after fillna: {null_total}")
print(f"Routes still ND after fix: {nd_paese}  (were 200 before the fix)")
print(f"Negative values: {neg_cols if neg_cols else 'none'}")


Null totali dopo fillna: 0
Rotte ancora ND dopo fix: 0  (erano 200 prima del fix)
Valori negativi: nessuno


## 7. Composite Risk Score

Combines the most significant features in a single weighted index [0, 1]:

| Component | Weight | Rationale |
|---|---|---|
| `tot_allarmi_log` normalised | 35% | Total alarm volume (log for outliers) |
| `score_rischio_esiti` | 35% | Security actions taken (rejected, detained) |
| `pct_interpol` | 15% | Fraction of INTERPOL alarms (international risk) |
| `tasso_rilevanza` | 15% | Relevant alarms on flights with alarms |


In [15]:
# ── New feature: alarm_per_invest ────────────────────────────────────────────
# From the partner notebook (03_Feature_Engineering.ipynb):
# Ratio of alerted / investigated per route.
# Measures alarm intensity on persons actually investigated.
# Capped at p99 to avoid extreme outliers (same approach as partner notebook).

agg_viaggiatori["alarm_per_invest"] = safe_div(
    agg_viaggiatori["tot_allarmati"],
    agg_viaggiatori["tot_investigati"]
)

# Cap at percentile 99 — extreme values are almost certainly encoding errors
cap_p99 = agg_viaggiatori["alarm_per_invest"].quantile(0.99)
agg_viaggiatori["alarm_per_invest"] = agg_viaggiatori["alarm_per_invest"].clip(upper=cap_p99)

print(f"alarm_per_invest: min={agg_viaggiatori['alarm_per_invest'].min():.4f}, "
      f"max={agg_viaggiatori['alarm_per_invest'].max():.4f} (cappato a p99={cap_p99:.4f}), "
      f"mean={agg_viaggiatori['alarm_per_invest'].mean():.4f}")

alarm_per_invest: min=0.0000, max=1.0000 (cappato a p99=1.0000), mean=0.2039


In [16]:
log_max = features["tot_allarmi_log"].max()

features["score_composito"] = (
    (features["tot_allarmi_log"] / max(log_max, 1)) * 0.35 +
    features["score_rischio_esiti"]                 * 0.35 +
    features["pct_interpol"]                        * 0.15 +
    features["tasso_rilevanza"]                     * 0.15
).clip(0, 1)

print("score_composito:")
print(features["score_composito"].describe().to_string())
print("\n── Top 10 routes by risk ─────────────────────────────────────────────")
top10_cols = ["ROTTA", "PAESE_PART", "score_composito",
              "tot_allarmi_log", "pct_interpol", "score_rischio_esiti", "tasso_rilevanza"]
top10 = (features[top10_cols]
         .sort_values("score_composito", ascending=False)
         .head(10))
print(top10.to_string(index=False))

score_composito:
count   567.0000
mean      0.2037
std       0.1637
min       0.0000
25%       0.0295
50%       0.2100
75%       0.3417
max       0.6620

── Top 10 rotte per rischio ──────────────────────────────────────────
  ROTTA          PAESE_PART  score_composito  tot_allarmi_log  pct_interpol  score_rischio_esiti  tasso_rilevanza
LHR-VCE         Regno Unito           0.6620           7.8586        0.4359               0.5333           0.5000
CMN-BLQ             Marocco           0.6346           7.0139        0.3043               0.4000           1.0000
SIN-MXP           Singapore           0.5949           7.2703        0.0000               0.6000           0.5000
ALG-MXP             Algeria           0.5486           5.1874        0.2500               0.4000           1.0000
TIA-BGY             Albania           0.5457           7.9738        0.1622               0.1613           0.8333
STN-BGY         Regno Unito           0.5423           8.1206        0.2716               0

In [17]:
# ── New feature: alarm_per_invest ────────────────────────────────────────────
# From the partner notebook (03_Feature_Engineering.ipynb):
# Ratio of alerted / investigated per route.
# Measures alarm intensity on persons actually investigated.
# Capped at p99 to avoid extreme outliers (same approach as partner notebook).

agg_viaggiatori["alarm_per_invest"] = safe_div(
    agg_viaggiatori["tot_allarmati"],
    agg_viaggiatori["tot_investigati"]
)

# Cap at percentile 99 — extreme values are almost certainly encoding errors
cap_p99 = agg_viaggiatori["alarm_per_invest"].quantile(0.99)
agg_viaggiatori["alarm_per_invest"] = agg_viaggiatori["alarm_per_invest"].clip(upper=cap_p99)

print(f"alarm_per_invest: min={agg_viaggiatori['alarm_per_invest'].min():.4f}, "
      f"max={agg_viaggiatori['alarm_per_invest'].max():.4f} (cappato a p99={cap_p99:.4f}), "
      f"mean={agg_viaggiatori['alarm_per_invest'].mean():.4f}")

alarm_per_invest: min=0.0000, max=1.0000 (cappato a p99=1.0000), mean=0.2039


## 8. Final quality report


In [18]:
sep = "=" * 62
print(sep)
print("  QUALITY REPORT — features_classical.csv")
print(sep)
print(f"\n  Shape: {features.shape[0]} rotte × {features.shape[1]} colonne")

# Null
nulls = features.isnull().sum()
nulls = nulls[nulls > 0]
print(f"\n  Null: {len(nulls)} colonne con null → {'✓ nessuno' if len(nulls)==0 else nulls.to_dict()}")

# Negativi
num = features.select_dtypes(include="number")
negs = {c: int((features[c] < 0).sum()) for c in num.columns if (features[c] < 0).sum() > 0}
print(f"  Negatives: {'none' if not negs else negs}")

# Riepilogo feature chiave
print("\n  Key feature statistics:")
key_feat = ["tot_allarmi_sum", "tot_allarmi_log", "pct_interpol", "pct_sdi",
            "tasso_chiusura", "tasso_rilevanza",
            "tasso_allarme_medio", "tasso_inv_medio",
            "score_rischio_esiti", "score_composito"]
print(features[key_feat].describe().T[["mean","std","min","max"]].to_string())

print(f"\n  Copertura rotte:")
print(f"    Both datasets:      {((features['n_osservazioni_allarmi']>0) & (features['n_osservazioni_viag']>0)).sum()}")
print(f"    Only ALLARMI:       {((features['n_osservazioni_allarmi']>0) & (features['n_osservazioni_viag']==0)).sum()}")
print(f"    Only VIAGGIATORI:   {((features['n_osservazioni_allarmi']==0) & (features['n_osservazioni_viag']>0)).sum()}")
print(sep)

  QUALITY REPORT — features_classical.csv

  Shape: 567 rotte × 62 colonne

  Null: 0 colonne con null → ✓ nessuno
  Negativi: ✓ nessuno

  Statistiche feature principali:
                        mean      std    min       max
tot_allarmi_sum     358.1605 664.3189 0.0000 3683.0000
tot_allarmi_log       3.1715   2.9648 0.0000    8.2118
pct_interpol          0.1150   0.1984 0.0000    1.0000
pct_sdi               0.1293   0.2137 0.0000    1.0000
tasso_chiusura        0.2441   0.4290 0.0000    1.0000
tasso_rilevanza       0.0682   0.2262 0.0000    1.0000
tasso_allarme_medio   0.1677   0.2751 0.0000    1.0000
tasso_inv_medio       0.7139   0.4075 0.0000    1.0000
score_rischio_esiti   0.1172   0.1656 0.0000    0.6000
score_composito       0.2037   0.1637 0.0000    0.6620

  Copertura rotte:
    Entrambi i dataset: 268
    Solo ALLARMI:       99
    Solo VIAGGIATORI:   199


In [19]:
# ── New feature: false_positive_rate ────────────────────────────────────────
# From the partner notebook (03_Feature_Engineering.ipynb):
# Measures how many alarms turned out to be false positives (no action taken).
# Denominator: SDI/NSIS + INTERPOL alarms (those verifiable)
# Numerator:    no-action SDI + NSIS + INT

# The necessary columns are already in occ_pivot after renaming
numeratore   = (agg_allarmi.get("np_sdi",   pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("np_nsis",  pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("np_int",   pd.Series(0, index=agg_allarmi.index)).clip(lower=0))

denominatore = (agg_allarmi.get("allarmi_sdi_occ",     pd.Series(0, index=agg_allarmi.index)).clip(lower=0) +
                agg_allarmi.get("allarmi_interpol_occ", pd.Series(0, index=agg_allarmi.index)).clip(lower=0))

agg_allarmi["false_positive_rate"] = safe_div(numeratore, denominatore).clip(0, 1)

print(f"false_positive_rate: min={agg_allarmi['false_positive_rate'].min():.4f}, "
      f"max={agg_allarmi['false_positive_rate'].max():.4f}, "
      f"mean={agg_allarmi['false_positive_rate'].mean():.4f}")
print(f"  Routes with FPR > 0.5: {(agg_allarmi['false_positive_rate'] > 0.5).sum()} "
      f"(high share of alarms that turned out to be false)")

false_positive_rate: min=0.0000, max=1.0000, mean=0.1276
  Rotte con FPR > 0.5: 43 (alta quota di allarmi rivelatisi falsi)


## 9. Saving


In [20]:
# Feature numeriche per i modelli ML
numeric_feature_cols = [
    c for c in features.select_dtypes(include="number").columns.tolist()
    if c != "score_composito"
]

meta = {
    "feature_cols" : numeric_feature_cols,
    "target"       : "score_composito",
    "n_routes"     : len(features),
    "n_features"   : len(numeric_feature_cols),
    "description"  : "Features aggregated by ROUTE — classical pipeline"
}

out_csv  = PROC_DIR / "features_classical.csv"
out_json = PROC_DIR / "feature_cols.json"

features.to_csv(out_csv, index=False)
with open(out_json, "w") as f:
    json.dump(meta, f, indent=2)

print(f"features_classical.csv  — {len(features)} routes × {features.shape[1]} columns")
print(f"feature_cols.json       — {len(numeric_feature_cols)} numerical features")
print(f"\nPercorso: {out_csv}")

✓ features_classical.csv  — 567 rotte × 62 colonne
✓ feature_cols.json       — 53 feature numeriche

Percorso: /Users/fil/Desktop/Machine learning condiviso /classical-vs-multiagent/data/processed/features_classical.csv


---
## Summary

| Step | Output | Dimensions |
|------|--------|----------|
| Pivot OCCORRENZE | `occ_pivot` | 368 routes × 31 col |
| % Alarm reason | `motivo_counts` | 351 routes × 6 col |
| ALLARMI metadata | `meta_allarmi` | 368 routes × 6 col |
| **agg_allarmi** | complete ALLARMI aggregation | **368 routes × 43 col** |
| VIAGGIATORI outcomes | `esiti_pivot` | 467 routes × 6 col |
| **agg_viaggiatori** | complete VIAGGIATORI aggregation | **467 routes × 20 col** |
| **features (outer join)** | final ML dataset | **567 routes × 63 col** |

### Next notebook
**`03_baseline_construction.ipynb`** — Building the historical baseline per route:
moving average, standard deviation, dynamic thresholds for anomaly detection.
